# **Lab:** Security in Computer Networks

**Student name:** Oscar Pinchao

### **Topics**
- Confidentiality
- Public Key Cryptography (RSA)
- Digital Signatures

In [30]:
# Install cryptography library (only first time)
# This library provides tools for encryption, decryption, signatures, etc.
%pip install cryptography

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
# Import RSA module for asymmetric cryptography
from cryptography.hazmat.primitives.asymmetric import rsa

# Import serialization tools (used for saving/loading keys, not used deeply here)
from cryptography.hazmat.primitives import serialization

# Import padding schemes (needed for secure encryption)
from cryptography.hazmat.primitives.asymmetric import padding

# Import hashing algorithms (SHA256, etc.)
from cryptography.hazmat.primitives import hashes

## **🔐 Part 1:** RSA (Public Key Cryptography)

- Public key → shared
- Private key → secret
- Encrypt with public key, decrypt with private key


In [32]:
# Generate PRIVATE key
# public_exponent=65537 is a standard safe value used in RSA
# key_size=2048 means 2048-bit key (secure for most purposes)
private_key = rsa.generate_private_key(
    public_exponent=65537,   # mathematical parameter used in RSA
    key_size=2048            # size of the key in bits (larger = more secure but slower)
)

# Generate PUBLIC key from private key
# In RSA, both keys are mathematically linked
public_key = private_key.public_key()

print("Keys generated successfully")

Keys generated successfully


In [33]:
# Export public key to PEM
pem_pub = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
).decode()

pem_pub

'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAo+kwzbJhl6zTVS27RCud\nikZTK3HZUWcZbObiGG2cnYvLHpvHcaiGrsArZDCmirLlo6xJkwG7zlHe1lVa9zgf\nOTg/UP0ydAsSAHsKq7mXybaB8HqXu/ACKTfrubygVvdtRQbUfFHyg2fjQQunJJ4Q\n7m1+9Rzu7HtEdaK8k9VJSEytQWq3qYns80DYUXqHABvnBYmg3jMDGRBdPtfdqtr3\nkQddWzBB/5e4ir9PpvwgJTVP2C+JJt1Tp4nA2W1EYh7HIZw1TxQy3xCZJZBfp+8D\nccnn9fyjOYEpKZv+x1lGB7MudpLZ2SBVsMBsEyevu82hrt5bJTOE9vvdPC/F/Kbe\nDQIDAQAB\n-----END PUBLIC KEY-----\n'

In [34]:
# Convert received PEM to an object
partner_pem = """
-----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAlopG7gNcUVYnmZHE554F
4RwZgdDzTvYjX68ntAAmuiOWABF8FkhUm+ks1oVNChgipk7AQwwMnLvLKASmIlGb
R9rTeIAWcloxYR1FHKJwp4QZY8HMdnSUKrH7U+nZ/WLbD0Ga/WA/BxZdbdKUya2Q
lRRA2KcMFuVi8VXQ6Y6hbHOaz0bxjYIBTiF0MzkJUrgmVikAkTBGvsBwlLhYA9cg
yPKH5gR9R/m1140vxNcnLXUcEQVhQipwoo6oj94U1vVFzt91LYoYFbpN3mG4Ym91
li+3Diz7jkAogIIkal3Dv+AW76waHP/cLOCx3tIa/xSo0ut4HPn/7dSQTSCgsD80
ewIDAQAB
-----END PUBLIC KEY-----
"""

received_public_key = serialization.load_pem_public_key(
    partner_pem.strip().encode()
)

received_public_key

In [35]:
import base64

# Define message to encrypt
# .encode() converts string → bytes (required for crypto functions)
message = "ola q hace".encode()

# Encrypt using PUBLIC key
encrypted = received_public_key.encrypt(
    message,  # data to encrypt (must be bytes)
    padding.OAEP(  # OAEP = Optimal Asymmetric Encryption Padding (secure padding)
        mgf=padding.MGF1(algorithm=hashes.SHA256()),  # MGF1 = Mask Generation Function
        algorithm=hashes.SHA256(),  # hash algorithm used internally
        label=None  # optional label (not needed here)
    )
)

print("Encrypted (Python bytes object):", encrypted) # This will show as a bytes object (not human-readable). ASCIIs are not suitable for binary data, so we will convert it to hex and base64 for display.
print("Encrypted (hex):", encrypted.hex())
print("Encrypted (base64):", base64.b64encode(encrypted).decode()) # Standard way to represent binary data as text

Encrypted (Python bytes object): b"[\xaf\xaaG\t\xfc[\x01M\xc1\x88\x05\x80g\xccg\xb0\xee\x1e\xf3(\x1e\xa4g\x82\xdf\xb5y\x8es^\xfe7\xdc\x98\xdb\xb6{9*z\xce.cE\xf6\xe5\x8dt=\xd8\xf0\x15\xd2\xff\x19\xf1\x17\x9a)f\x95#\xd2'4\xcb\xaf]\x1f\xd7\xac\xa2\x00\xebwmy\xf1\x95\xe7\x19\xf2 Q\xfb\xf9\xf9H~q\xaa\xad\xdd\x0f\xc6\x13 k\x8a&<\xc0f\x1d\xd1`\x11B\\\x1fYHw\xed\x03\xc0\xd1R\xd2\t\xc1_8\xd2\\\xf3\x8cA\x86\xed\xe3\x8e\xc8\xcc\xb7\xc9\x7f>C\x87\xe9'L\xbe@Ba\xce\xbe\xce\t<\xdf-\x97\t\r\x91ln\x9b\xe7\xedb$\xf5\x97`\xbe}\x061\xc1~V\xc5\x88\xbb\x9b\xe1\x08\x8e\x897\x1c.\xb1\x16yGz\xdc>\x17\xcb\x8c>M~7~,\xe4\xba\xa7!bV#\xe4\n\xc1I\xbf\xe9\xe1\x9c\x981I1\xd9\x1b\x1d\xf3\xf4aki\xb5,\x15\x13%Va\xf1?T\xc8\xa5\xf7(\xe0/\xd9,1\n\xf3\xb8W\xf3\xd87"
Encrypted (hex): 5bafaa4709fc5b014dc188058067cc67b0ee1ef3281ea46782dfb5798e735efe37dc98dbb67b392a7ace2e6345f6e58d743dd8f015d2ff19f1179a29669523d22734cbaf5d1fd7aca200eb776d79f195e719f22051fbf9f9487e71aaaddd0fc613206b8a263cc0661dd16011425c1f594877ed03c0d152d209c15f

In [36]:
received_encrypted_b64 = """
HT6QYXvkmYmDvH6490Mh+mpWlSPsI9Z6UO3MAztGAEqeRbWu/AZVYiK5qHzdCunwrporOb07qX1oeMf11hY86Vi3b1T4URAPUIfY39icas1yp06DqsNOw0wHq21s+A5Te6FuIDUp/SpMEq37h05LBkmWp/8qKM+WvxaZgYSGFb/3LitOhxq3lgG0fzL2ImlBDJd4qb7b/34yyTYjG56ISegrzs+Yki24BN7NeCyjPuPP7tOsnTmeMuutBOrnCw8/+besv3BQH0XbWv52GVDqLZBV2PbUg08EeAG/9Va7XCo53AY1DNvUNL0BksSoMXHdVDVHR1/3w3MPwSS/NuG+Nw==
"""

encrypted = base64.b64decode(
    "".join(received_encrypted_b64.split()),
    validate=True
)
# Decrypt using PRIVATE key
decrypted = private_key.decrypt(
    encrypted,  # encrypted data
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Convert bytes → string using .decode()
print("Decrypted:", decrypted.decode())

ValueError: Decryption failed

When you:

1. Take a message  
2. Encrypt it with the **receiver’s public key**  
3. Encode it in **Base64**

You already achieved **confidentiality**:  
Only the holder of the **private key** can decrypt it.

> ⚠️ **Base64 is NOT security**, it is just an encoding format to represent binary data as text.

The real problem (without certificates)

The key question is:

> ❗ How do you know that the **public key** actually belongs to the right person?

Without certificates, this can happen:

- An attacker gives you **their public key**
- You encrypt the message using that key
- The attacker decrypts it with their private key
- Then re-encrypts it using the real recipient’s key

*Result:* your communication was intercepted without you noticing.

This is known as a **Man-in-the-Middle** attack.

So… what are certificates for?

A certificate (e.g., `X.509`) is used to:

**Bind:**
- A **public key**
- To a **real identity** (person, server, organization)

**It also includes:**
- A trusted signature (from a Certificate Authority)
- Owner information
- Expiration/validity dates

Who guarantees this?

A **Certificate Authority (CA)** such as:

- Let's Encrypt  
- DigiCert  

These entities sign certificates and essentially say:

> “Yes, this public key belongs to this entity.”

## **✍️ Part 2:** Digital Signatures

- Sign with PRIVATE key
- Verify with PUBLIC key


In [ ]:
# Create digital signature
signature = private_key.sign(
    message,  # original message
    padding.PSS(  # Probabilistic Signature Scheme (secure signature padding)
        mgf=padding.MGF1(hashes.SHA256()),  # mask generation function
        salt_length=padding.PSS.MAX_LENGTH  # random salt for security
    ),
    hashes.SHA256()  # hash algorithm used before signing
)

print("Signature created")
print("Signature (base64):", base64.b64encode(signature).decode())


Signature created
Signature (base64): Z2/ko3Qw5yjVZtCZqjOLv/+NRRGNJ6ohnfA7qQoS+HfnRS/OimAmuSLOzzX5eohou7TunIYIA6qLHtzxwHboz2yrfrwnoSwwwCCBgOUCnzTtNxjCnK7aXXO8V4ZA7LFJTZzPsFL2TFcxt1Zz27OGfFqdphmi3qHT8oVLMl49RbH1VjYEE0BoFnEOWum2MjJyEiO+OkTc67N7u9tIVF82s8HCGSDoHg5/QOxmyVXJPDN1B/5p8xNqyx9+5BqyY9VAhrhfCOhd0fSDEA+swRJktaOs4GFyFMLEtpYnnAeHq82FRtRxySzoltoN30SR6pWBzAzpyVk8Je6WBsuqSmDCsg==


In [ ]:
received_signature_b64 = """
Lq8I2LwfIivB/4Wox6UBRyyiIINcV/FsiO0S2J+WqxqRZD3dGTN7aafTKN3fTyvHx1Uf13dTdvh2VfV08GNfIgUxXDkvOxOBZdAG+z1O9gghbv1Sb8CYAARaiQL+OT2svYnff7N1lHGdTFJnPelVAzU6qS152UqMoQLhVFXfx5fEdzQWP6aN2HuX+N8advc0R51o5O41RPpDsDbNxeijQctoFbBlL/c5H4J7Co1VXXAzuMeTx51fex8b5yHPAAOgWZ2Amm2VS1xOnTtIKTv6d5k2ziUqzAjAPIi85gW6APz3IN4Ikh0jeKefZsDAL6iH/whKLVlr4GzGzl8H7FlKxg=="""

signature = base64.b64decode(
    "".join(received_signature_b64.split()),
    validate=True
)
# Verify signature
try:
    received_public_key.verify(
        signature,
        decrypted,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )
    print("Signature VALID")
except:
    print("Signature INVALID")

Signature VALID


How it fits together?

In protocols like **Transport Layer Security (TLS)**:

1. The server sends its certificate  
2. The client verifies:
   - The CA signature  
   - The domain name  
3. **Only then** the public key is trusted  
4. The key is used for encryption

## **🌐 Part 3:** TLS Handshake Simulation

Real TLS:
1. Client gets server public key
2. Client creates session key
3. Client encrypts session key with public key
4. Server decrypts
5. Both use symmetric encryption


In [ ]:
# Import symmetric encryption tool
from cryptography.fernet import Fernet

In [ ]:
# Client generates symmetric session key
# This key will be used for fast encryption (like AES)
session_key = Fernet.generate_key()

# Create cipher object for encryption/decryption
cipher_client = Fernet(session_key)

print("Session key created")

Session key created


In [ ]:
# Client encrypts session key using server PUBLIC key
encrypted_session_key = public_key.encrypt(
    session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

In [ ]:
# Server decrypts session key using PRIVATE key
decrypted_session_key = private_key.decrypt(
    encrypted_session_key,
    padding.OAEP(
        mgf=padding.MGF1(hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)

# Server creates cipher with shared key
cipher_server = Fernet(decrypted_session_key)


In [ ]:
# Secure communication
secure_message = cipher_client.encrypt(b"Hello TLS secure world")

print("Encrypted message:", secure_message)

print("Decrypted message:", cipher_server.decrypt(secure_message).decode())

Encrypted message: b'gAAAAABq-0YLHKr9_pEx200n8kINHoc_5V1527Wsho_Yyxsd2t3M2FVzflMgZZ7a3Lt64qXVMW6GyRt1jo_G22DsNQCHv3-LsxIRHQElotHhbfg0YUxtklo='
Decrypted message: Hello TLS secure world


## **Assignment:** Secure Communication Challenge

### **Objective**
Apply all concepts from the lab:
- Confidentiality (encryption)
- Authentication (digital signatures)
- Integrity (verification)

### **Group Dynamics (Balanced Work)**

Each student must complete ALL roles:

1. **Sender**
2. **Receiver**

You will interact with at least **one other student**.

### **Task 1:** Secure Message Sending

Each student must:

1. Generate an RSA key pair
2. Share ONLY the **public key**
3. Create a message
4. Encrypt the message using the receiver’s public key
5. Sign the message using your private key

👉 Send to another student:
- Encrypted message
- Signature
- Your public key

### **Task 2:** Secure Message Receiving

Each student must:

1. Receive:
   - Encrypted message
   - Signature
   - Sender's public key

2. Decrypt the message using your private key  
3. Verify the signature using the sender’s public key  

✔ If valid → accept message  
❌ If invalid → reject message  

### **Time Limit**
- 20–25 minutes total

### **Evaluation (Checklist)**

| **Criteria** | **Score (10pts)** |
|--------|------|
| RSA encryption implemented | 1 pts |
| Digital signature implemented | 1 pts |
| Correctly decrypted & verified message | 4 pts |
| Explanation reflection questions | 4 pts |

### **Reflection Questions**

1. Why is encryption alone NOT enough for security? 

Encryption keeps the message private, but it does not prove who sent it. In this lab, anyone with my public key could encrypt a message for me. That is why I also needed to verify the digital signature and make sure the public key belonged to my partner. 

2. What is the role of digital signatures?

Digital signatures help check who signed a message and whether it was changed. I signed my message with my private key, and my partner could verify it with my public key. When I received my partner’s message, I used their public key to verify the signature. The result was “Signature VALID,” which showed that the received message matched the signature. 

3. Why does TLS use a hybrid approach?  

TLS combines asymmetric and symmetric cryptography because they have different advantages. Asymmetric cryptography helps with authentication and establishing shared keys, while symmetric encryption is faster for exchanging data. In the lab simulation, RSA protected the session key, and then Fernet used that key to encrypt and decrypt the message.